In [3]:
"""
Iceberg Mass & Uncertainty (95% CI half-width) Writer
=====================================================

This script reads yearly Antarctic iceberg shapefiles, repairs geometries,
derives geometric attributes in a polar stereographic projection (EPSG:3031),
and writes a GeoPackage (GPKG) per year containing ONLY the following columns:

    lon, lat, area_km2, area_uncertainty_km2, perimeter_km,
    long_axis_km, short_axis_km, mass_gt, mass_uncertainty_gt
"""

import os
import glob
import shutil
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import MultiPolygon
from shapely import minimum_rotated_rectangle
try:
    # Prefer Shapely >= 2.0 geometry repair if available
    from shapely.validation import make_valid as shapely_make_valid
    HAS_MAKE_VALID = True
except Exception:
    HAS_MAKE_VALID = False

# =========================
# User configuration
# =========================
CALIB_XLSX = "data.xlsx"                       # Calibration workbook (A, V)
CALIB_SHEET = 0                                # Sheet index or name
COL_A_CANDIDATES = ["Shape_Area", "Shape_Are"] # Area column candidates (m²)
COL_V = "volume_all"                           # Volume column name (m³)
N_BOOT = 5000                                  # Number of bootstrap draws
SEED = 42                                      # RNG seed for reproducibility

SRC_DIR = "."                                  # Parent dir containing yearly folders
OUT_VEC_DIR = "Iceberg vector outline"         # Destination dir for copied GPKGs

# =========================
# Physical / algorithmic constants
# =========================
RHO_ICE = 850.0            # Ice density (kg/m³)
H_LARGE = 250.0            # Fixed thickness for large icebergs (m)
AREA_SPLIT_KM2 = 0.67      # Area split threshold (km²)
KM2_TO_M2 = 1e6            # km² → m²
KG_TO_GT  = 1e-12          # kg → Gt
DX_KM = 0.04               # Spatial resolution used for area uncertainty (km)
ANTARCTIC_EPSG = 3031      # Antarctic Polar Stereographic (meters)

warnings.filterwarnings("ignore", category=UserWarning)
os.makedirs(OUT_VEC_DIR, exist_ok=True)

# ---------------------------------------------------------------------
# Geometry utilities
# ---------------------------------------------------------------------
def fix_geometry(geom):
    """Repair invalid/empty geometries with make_valid (if available) then buffer(0)."""
    if geom is None or geom.is_empty:
        return None
    if geom.is_valid:
        return geom
    if HAS_MAKE_VALID:
        g2 = shapely_make_valid(geom)
        if g2 is not None and g2.is_valid:
            return g2
    try:
        g2 = geom.buffer(0)
        if g2.is_valid:
            return g2
    except Exception:
        pass
    return None

def largest_polygon(geom):
    """For MultiPolygon, return the largest component; otherwise return original."""
    if isinstance(geom, MultiPolygon):
        if len(geom.geoms) == 0:
            return None
        return max(geom.geoms, key=lambda g: g.area)
    return geom

def axes_from_min_rot_rect(geom_m):
    """
    Compute long/short axes (km) from the minimum rotated rectangle in EPSG:3031.
    """
    poly = largest_polygon(geom_m)
    if poly is None or poly.is_empty or poly.area <= 0.0:
        return 0.0, 0.0
    mrr = minimum_rotated_rectangle(poly)
    coords = np.asarray(mrr.exterior.coords)
    if coords.shape[0] < 5:
        return 0.0, 0.0
    edges = np.linalg.norm(np.diff(coords, axis=0), axis=1)  # meters
    long_axis_km = float(np.max(edges)) / 1000.0
    short_axis_km = float(np.min(edges)) / 1000.0
    if short_axis_km > long_axis_km:
        long_axis_km, short_axis_km = short_axis_km, long_axis_km
    return long_axis_km, short_axis_km

def year_from_fname(path: str) -> str:
    """Extract first 4 digits (YYYY) from a path or filename."""
    base = os.path.basename(path)
    return base[:4] if base[:4].isdigit() else "NA"

# ---------------------------------------------------------------------
# Calibration (power-law fit in log-space) and bootstrap
# ---------------------------------------------------------------------
def fit_ln_model(A_m2: np.ndarray, V_m3: np.ndarray):
    """Fit ln V = b0 + b1 ln A via least squares."""
    x = np.log(A_m2)
    y = np.log(V_m3)
    X = np.vstack([np.ones_like(x), x]).T
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    return float(beta[0]), float(beta[1])

def bootstrap_params(A_m2: np.ndarray, V_m3: np.ndarray, n_boot=5000, seed=42):
    """Bootstrap (b0, b1) by resampling calibration pairs (A, V) with replacement."""
    rng = np.random.default_rng(seed)
    n = len(A_m2)
    B0 = np.empty(n_boot)
    B1 = np.empty(n_boot)
    idx_all = np.arange(n)
    for i in range(n_boot):
        idx = rng.choice(idx_all, size=n, replace=True)
        b0, b1 = fit_ln_model(A_m2[idx], V_m3[idx])
        B0[i], B1[i] = b0, b1
    return B0, B1

def calibrate():
    """
    Load calibration data, pick columns, clean, fit central (b0, b1),
    and return bootstrap samples.
    """
    if not os.path.exists(CALIB_XLSX):
        raise FileNotFoundError(f"Calibration file not found: {CALIB_XLSX}")
    dcal = pd.read_excel(CALIB_XLSX, sheet_name=CALIB_SHEET)
    col_a = next((c for c in COL_A_CANDIDATES if c in dcal.columns), None)
    if col_a is None:
        raise KeyError(f"Area column missing; provide one of: {COL_A_CANDIDATES}")
    if COL_V not in dcal.columns:
        raise KeyError(f"Volume column '{COL_V}' not found.")
    A_m2 = pd.to_numeric(dcal[col_a], errors="coerce").to_numpy(dtype=float)
    V_m3 = pd.to_numeric(dcal[COL_V], errors="coerce").to_numpy(dtype=float)
    msk = np.isfinite(A_m2) & np.isfinite(V_m3) & (A_m2 > 0) & (V_m3 > 0)
    A_m2, V_m3 = A_m2[msk], V_m3[msk]
    b0_cen, b1_cen = fit_ln_model(A_m2, V_m3)
    B0_samp, B1_samp = bootstrap_params(A_m2, V_m3, n_boot=N_BOOT, seed=SEED)
    return b0_cen, b1_cen, B0_samp, B1_samp

# ---------------------------------------------------------------------
# Main processing
# ---------------------------------------------------------------------
def main():
    # 1) Calibrate the small-berg power-law and obtain H distribution at the split
    b0_cen, b1_cen, B0_samp, B1_samp = calibrate()

    # Equivalent thickness distribution at the split:
    # H_boot = V(A_split)/A_split = [exp(b0)*A_split^b1] / A_split
    A_split_m2 = AREA_SPLIT_KM2 * KM2_TO_M2
    H_boot = (np.exp(B0_samp) * (A_split_m2 ** B1_samp)) / A_split_m2  # meters
    H_lo, H_hi = np.quantile(H_boot, [0.025, 0.975])

    # 2) Collect yearly shapefiles "<year>/<year>10_distribution.shp"
    shp_paths = []
    for year in range(2018, 2024):
        shp = os.path.join(SRC_DIR, str(year), f"{year}10_distribution.shp")
        if os.path.exists(shp):
            shp_paths.append(shp)
        else:
            print(f"[WARN] Missing {shp}; skipping.")

    # 3) Process each year
    for shp in shp_paths:
        year = year_from_fname(shp)
        gpkg = os.path.join(SRC_DIR, year, f"{year}10_distribution.gpkg")
        layer = f"layer_{year}10_distribution"

        # Read & repair
        gdf_src = gpd.read_file(shp)
        gdf_src["geometry"] = gdf_src["geometry"].apply(fix_geometry)
        gdf_src = gdf_src[~gdf_src.geometry.isna() & ~gdf_src.geometry.is_empty].copy()

        # lon/lat: use existing columns if present, else centroid in WGS84
        if "lon" in gdf_src.columns and "lat" in gdf_src.columns:
            lon = pd.to_numeric(gdf_src["lon"], errors="coerce")
            lat = pd.to_numeric(gdf_src["lat"], errors="coerce")
        else:
            g_ll = gdf_src.to_crs(4326)
            cen = g_ll.geometry.centroid
            lon = pd.Series(cen.x, index=gdf_src.index)
            lat = pd.Series(cen.y, index=gdf_src.index)

        # area_km2 / perimeter_km: prefer existing; recompute missing in EPSG:3031
        if "area" in gdf_src.columns:
            area_km2 = pd.to_numeric(gdf_src["area"], errors="coerce")
        else:
            area_km2 = pd.Series(np.nan, index=gdf_src.index)

        if "perimeter" in gdf_src.columns:
            perimeter_km = pd.to_numeric(gdf_src["perimeter"], errors="coerce")
        else:
            perimeter_km = pd.Series(np.nan, index=gdf_src.index)

        # Compute any missing values in metric CRS
        need_recalc = area_km2.isna() | perimeter_km.isna()
        gdf_m = gdf_src.to_crs(ANTARCTIC_EPSG)
        if need_recalc.any():
            area_km2.loc[need_recalc] = gdf_m.geometry.area[need_recalc] / 1e6
            perimeter_km.loc[need_recalc] = gdf_m.geometry.length[need_recalc] / 1e3

        # Long/short axes (km) from minimum rotated rectangle
        axes = gdf_m.geometry.apply(axes_from_min_rot_rect)
        long_axis_km, short_axis_km = zip(*axes)

        # -------- Area uncertainty per iceberg: ONLY U1 (boundary band) --------
        # U1_i = perimeter_km * DX_KM  (units: km * km = km²)
        area_uncertainty_km2 = perimeter_km * DX_KM

        # Two-segment mass & uncertainty
        A_km2 = area_km2.to_numpy(dtype=float)
        A_m2 = A_km2 * KM2_TO_M2
        is_small = A_km2 < AREA_SPLIT_KM2
        is_large = ~is_small

        # Central mass (Gt)
        mass_gt = np.zeros(len(gdf_src))
        if np.any(is_small):
            V_small = np.exp(b0_cen) * np.power(A_m2[is_small], b1_cen)     # m³
            mass_gt[is_small] = V_small * RHO_ICE * KG_TO_GT               # Gt
        if np.any(is_large):
            mass_gt[is_large] = A_m2[is_large] * H_LARGE * RHO_ICE * KG_TO_GT

        # 95% CI half-width (Gt)
        mass_uncertainty_gt = np.zeros(len(gdf_src))

        # Small: bootstrap per iceberg
        if np.any(is_small):
            for idx, a in zip(np.where(is_small)[0], A_m2[is_small]):
                m_samp_gt = (np.exp(B0_samp) * (a ** B1_samp)) * RHO_ICE * KG_TO_GT
                lo, hi = np.quantile(m_samp_gt, [0.025, 0.975])
                mass_uncertainty_gt[idx] = (hi - lo) / 2.0

        # Large: shared thickness distribution scaled by area
        if np.any(is_large):
            lo = A_m2[is_large] * np.quantile(H_boot, 0.025) * RHO_ICE * KG_TO_GT
            hi = A_m2[is_large] * np.quantile(H_boot, 0.975) * RHO_ICE * KG_TO_GT
            mass_uncertainty_gt[is_large] = (hi - lo) / 2.0

        # Assemble output (kept in EPSG:3031 for geometry)
        out = gdf_m.copy()
        out["lon"] = lon.values
        out["lat"] = lat.values
        out["area_km2"] = area_km2.values
        out["area_uncertainty_km2"] = area_uncertainty_km2.values
        out["perimeter_km"] = perimeter_km.values
        out["long_axis_km"] = np.asarray(long_axis_km)
        out["short_axis_km"] = np.asarray(short_axis_km)
        out["mass_gt"] = mass_gt
        out["mass_uncertainty_gt"] = mass_uncertainty_gt

        keep_cols = [
            "lon", "lat",
            "area_km2", "area_uncertainty_km2", "perimeter_km",
            "long_axis_km", "short_axis_km",
            "mass_gt", "mass_uncertainty_gt",
            "geometry"
        ]
        out = out[keep_cols]

        # Write GPKG (single layer) and copy to OUT_VEC_DIR
        out.to_file(gpkg, layer=layer, driver="GPKG")
        shutil.copy(gpkg, os.path.join(OUT_VEC_DIR, os.path.basename(gpkg)))
        print(f"{year} → wrote columns {keep_cols[:-1]} to {gpkg}")

if __name__ == "__main__":
    main()


2018 → wrote columns ['lon', 'lat', 'area_km2', 'area_uncertainty_km2', 'perimeter_km', 'long_axis_km', 'short_axis_km', 'mass_gt', 'mass_uncertainty_gt'] to .\2018\201810_distribution.gpkg
2019 → wrote columns ['lon', 'lat', 'area_km2', 'area_uncertainty_km2', 'perimeter_km', 'long_axis_km', 'short_axis_km', 'mass_gt', 'mass_uncertainty_gt'] to .\2019\201910_distribution.gpkg
2020 → wrote columns ['lon', 'lat', 'area_km2', 'area_uncertainty_km2', 'perimeter_km', 'long_axis_km', 'short_axis_km', 'mass_gt', 'mass_uncertainty_gt'] to .\2020\202010_distribution.gpkg
2021 → wrote columns ['lon', 'lat', 'area_km2', 'area_uncertainty_km2', 'perimeter_km', 'long_axis_km', 'short_axis_km', 'mass_gt', 'mass_uncertainty_gt'] to .\2021\202110_distribution.gpkg
2022 → wrote columns ['lon', 'lat', 'area_km2', 'area_uncertainty_km2', 'perimeter_km', 'long_axis_km', 'short_axis_km', 'mass_gt', 'mass_uncertainty_gt'] to .\2022\202210_distribution.gpkg
2023 → wrote columns ['lon', 'lat', 'area_km2', 'a

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd

# ==============================
# Iceberg Mass Summary (New Algorithm with Bootstrap CI)
# Calibration data: data.xlsx (columns: Shape_Area or Shape_Are [m²], volume_all [m³])
#
# New algorithm:
#   - Small icebergs (A < 0.67 km²): volume from power-law regression
#   - Large icebergs (A ≥ 0.67 km²): fixed thickness H = 250 m
#
# Uncertainty (95% CI):
#   - For each bootstrap sample, compute small iceberg mass + large iceberg mass
#   - The empirical distribution of total mass is used to derive CI
#
# Area uncertainty:
#   - U1 = total_perimeter * DX
#   - U2 = p_detect * total_area         (e.g., 0.04 = 4%)
#   - U3 = p_mosaic * total_area         (e.g., 0.02 = 2%)
#   - UA = sqrt(U1^2 + U2^2 + U3^2)
# ==============================

# -------- Paths and files --------
GPKG_DIR = "Iceberg vector outline"
CALIB_XLSX = "data.xlsx"
CALIB_SHEET = 0
COL_A_CANDIDATES = ["Shape_Area", "Shape_Are"]
COL_V = "volume_all"

# -------- Constants --------
N_BOOT = 5000
RHO_ICE = 850.0            # kg/m³
H_LARGE = 250.0            # m (new algorithm, large iceberg thickness)
H_OLD   = 232.0            # m (old algorithm, not printed here)
AREA_SPLIT_KM2 = 0.67      # threshold (km²)
KM2_TO_M2 = 1e6
KG_TO_GT  = 1e-12
ANTARCTIC_EPSG = 3031
DX_KM = 0.04               # km, used for U1 (40 m)

# Area-uncertainty proportions
P_DETECT = 0.04            # U2 = 4% of total area (detection/segmentation)
P_MOSAIC = 0.02            # U3 = 2% of total area (mosaic/duplicate-related)

# -------- Calibration (ln V = b0 + b1 ln A) --------
def fit_ln_model(A_m2: np.ndarray, V_m3: np.ndarray):
    x = np.log(A_m2)
    y = np.log(V_m3)
    X = np.vstack([np.ones_like(x), x]).T
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    return float(beta[0]), float(beta[1])

def bootstrap_params(A_m2: np.ndarray, V_m3: np.ndarray, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(A_m2)
    B0 = np.empty(n_boot)
    B1 = np.empty(n_boot)
    idx_all = np.arange(n)
    for i in range(n_boot):
        idx = rng.choice(idx_all, size=n, replace=True)
        b0, b1 = fit_ln_model(A_m2[idx], V_m3[idx])
        B0[i], B1[i] = b0, b1
    return B0, B1

def load_calibration():
    if not os.path.exists(CALIB_XLSX):
        raise FileNotFoundError(f"Calibration file not found: {CALIB_XLSX}")
    dcal = pd.read_excel(CALIB_XLSX, sheet_name=CALIB_SHEET)
    col_a = next((c for c in COL_A_CANDIDATES if c in dcal.columns), None)
    if col_a is None:
        raise KeyError(f"Area column missing. Provide one of {COL_A_CANDIDATES}")
    if COL_V not in dcal.columns:
        raise KeyError(f"Volume column '{COL_V}' missing in calibration file")

    A_m2 = pd.to_numeric(dcal[col_a], errors="coerce").to_numpy(dtype=float)
    V_m3 = pd.to_numeric(dcal[COL_V], errors="coerce").to_numpy(dtype=float)
    msk = np.isfinite(A_m2) & np.isfinite(V_m3) & (A_m2 > 0) & (V_m3 > 0)
    A_m2, V_m3 = A_m2[msk], V_m3[msk]

    b0_cen, b1_cen = fit_ln_model(A_m2, V_m3)
    B0_samp, B1_samp = bootstrap_params(A_m2, V_m3, n_boot=N_BOOT, seed=42)
    return b0_cen, b1_cen, B0_samp, B1_samp

# -------- Area retrieval --------
def ensure_area_km2(gdf: gpd.GeoDataFrame) -> np.ndarray:
    if "area_km2" in gdf.columns:
        return pd.to_numeric(gdf["area_km2"], errors="coerce").fillna(0).to_numpy()
    if "area" in gdf.columns:  # some files may use "area"
        return pd.to_numeric(gdf["area"], errors="coerce").fillna(0).to_numpy()
    gdf_m = gdf if (gdf.crs and gdf.crs.to_epsg() == ANTARCTIC_EPSG) else gdf.to_crs(ANTARCTIC_EPSG)
    return gdf_m.geometry.area.to_numpy() * 1e-6

# -------- Core function: summarize one year --------
def summarize_and_print(gpkg_path, b0_cen, b1_cen, B0_samp, B1_samp):
    gdf = gpd.read_file(gpkg_path)

    if "perimeter_km" not in gdf.columns:
        gdf_m = gdf if (gdf.crs and gdf.crs.to_epsg() == ANTARCTIC_EPSG) else gdf.to_crs(ANTARCTIC_EPSG)
        gdf["perimeter_km"] = gdf_m.geometry.length * 1e-3

    areas_km2 = ensure_area_km2(gdf)
    perimeter_km = pd.to_numeric(gdf["perimeter_km"], errors="coerce").fillna(0.0).to_numpy()

    year = os.path.basename(gpkg_path)[:4]
    number = int(len(gdf))
    total_area = float(np.nansum(areas_km2))
    total_perimeter = float(np.nansum(perimeter_km))

    # Area uncertainty components
    U1_total = total_perimeter * DX_KM              # km²
    U2_total = P_DETECT * total_area                # km²
    U3_total = P_MOSAIC * total_area                # km²
    UA = float(np.sqrt(U1_total**2 + U2_total**2 + U3_total**2))

    max_indiv = float(np.nanmax(perimeter_km * DX_KM)) if number > 0 else 0.0

    # Mass central estimate
    A = areas_km2
    mask_small = A < AREA_SPLIT_KM2
    A_small_km2 = A[mask_small]
    A_large_km2 = A[~mask_small]

    mass_small_new_kg = 0.0
    if A_small_km2.size:
        A_small_m2 = A_small_km2 * KM2_TO_M2
        V_small = np.exp(b0_cen) * np.power(A_small_m2, b1_cen)
        mass_small_new_kg = float(np.sum(V_small) * RHO_ICE)
    mass_large_new_kg = float(np.sum(A_large_km2) * KM2_TO_M2 * H_LARGE * RHO_ICE)
    mass_Gt = (mass_small_new_kg + mass_large_new_kg) * KG_TO_GT

    # Bootstrap total mass distribution
    n_boot = len(B0_samp)
    M_small_boot_kg = np.zeros(n_boot)
    M_large_boot_kg = np.zeros(n_boot)

    if A_small_km2.size:
        A_small_m2 = A_small_km2 * KM2_TO_M2
        # sum(A^b1) per bootstrap b1
        sums_pow = np.array([np.sum(np.power(A_small_m2, b1)) for b1 in B1_samp])
        M_small_boot_kg = np.exp(B0_samp) * sums_pow * RHO_ICE
    if A_large_km2.size:
        A_large_m2_sum = np.sum(A_large_km2) * KM2_TO_M2
        A_split_m2 = AREA_SPLIT_KM2 * KM2_TO_M2
        # derive an effective thickness at the split area from the bootstrap params
        H_boot = (np.exp(B0_samp) * (A_split_m2 ** B1_samp)) / A_split_m2
        M_large_boot_kg = A_large_m2_sum * H_boot * RHO_ICE

    M_total_boot_kg = M_small_boot_kg + M_large_boot_kg
    mt_lo_kg, mt_hi_kg = np.quantile(M_total_boot_kg, [0.025, 0.975])
    mass_uncertainty_Gt = 0.5 * (mt_hi_kg - mt_lo_kg) * KG_TO_GT
    rel_err = 100.0 * mass_uncertainty_Gt / mass_Gt if mass_Gt > 0 else np.nan

    # Print required output
    print(f"Year: {year}")
    print(f"number: {number}")
    print(f" Total area: {total_area:.4f} km²")
    print(f"  - U1 (perimeter×DX): {U1_total:.4f} km²")
    print(f"  - U2 (detect {P_DETECT*100:.0f}% of area): {U2_total:.4f} km²")
    print(f"  - U3 (mosaic {P_MOSAIC*100:.0f}% of area): {U3_total:.4f} km²")
    print(f" Combined area uncertainty UA: {UA:.4f} km²")
    print(f" Max individual U1: {max_indiv:.4f} km²")
    print(f" Total mass M: {mass_Gt:.4f} Gt")
    print(f" Mass uncertainty: {mass_uncertainty_Gt:.4f} Gt")
    print(f" Mass error ratio: {rel_err:.4f}%")
    print("--------------------------------------------------")

def main():
    b0_cen, b1_cen, B0_samp, B1_samp = load_calibration()
    gpkg_files = sorted(glob.glob(os.path.join(GPKG_DIR, "*.gpkg")))
    if not gpkg_files:
        print(f"No GPKG files found in {GPKG_DIR}")
        return
    for gpkg in gpkg_files:
        summarize_and_print(gpkg, b0_cen, b1_cen, B0_samp, B1_samp)

if __name__ == "__main__":
    main()


Year: 2018
number: 34825
 Total area: 38667.6560 km²
  - U1 (perimeter×DX): 4311.5762 km²
  - U2 (detect 4% of area): 1546.7062 km²
  - U3 (mosaic 2% of area): 773.3531 km²
 Combined area uncertainty UA: 4645.4348 km²
 Max individual U1: 20.0664 km²
 Total mass M: 7895.1159 Gt
 Mass uncertainty: 1666.8954 Gt
 Mass error ratio: 21.1130%
--------------------------------------------------
Year: 2019
number: 39261
 Total area: 42000.5053 km²
  - U1 (perimeter×DX): 4744.8493 km²
  - U2 (detect 4% of area): 1680.0202 km²
  - U3 (mosaic 2% of area): 840.0101 km²
 Combined area uncertainty UA: 5103.1049 km²
 Max individual U1: 22.2374 km²
 Total mass M: 8563.3829 Gt
 Mass uncertainty: 1806.1011 Gt
 Mass error ratio: 21.0910%
--------------------------------------------------
Year: 2020
number: 38066
 Total area: 45958.7320 km²
  - U1 (perimeter×DX): 4840.1623 km²
  - U2 (detect 4% of area): 1838.3493 km²
  - U3 (mosaic 2% of area): 919.1746 km²
 Combined area uncertainty UA: 5258.4771 km²
 Max

In [8]:
import os
from glob import glob
import pandas as pd
import geopandas as gpd

# Folder containing the GeoPackage files
gpkg_folder = "Iceberg vector outline"

# Define size bins (km²) and labels
bins = [0, 1, 10, 100, 1000, float('inf')]
labels = ['0-1 km²', '1-10 km²', '10-100 km²', '100-1000 km²', '>1000 km²']

# Collect results in a list
data = []

# List all GeoPackage files in the folder
gpkg_paths = glob(os.path.join(gpkg_folder, '*.gpkg'))

for gpkg in sorted(gpkg_paths):
    # Extract year from filename (assumes filename starts with YYYY)
    year = int(os.path.basename(gpkg)[:4])

    # Read the GeoPackage layer (assumes single layer per file)
    layer_name = f"layer_{year}10_distribution"
    try:
        gdf = gpd.read_file(gpkg, layer=layer_name)
    except Exception:
        gdf = gpd.read_file(gpkg)

    # Ensure numeric
    gdf['area_km2'] = pd.to_numeric(gdf['area_km2'], errors="coerce").fillna(0.0)
    gdf['mass_gt']  = pd.to_numeric(gdf['mass_gt'], errors="coerce").fillna(0.0)

    # Categorize each iceberg by area (km²)
    gdf['Category'] = pd.cut(gdf['area_km2'], bins=bins, labels=labels, right=False)

    # Sum mass per category in Gt
    grouped = gdf.groupby('Category')['mass_gt'].sum().to_dict()
    total_mass = float(gdf['mass_gt'].sum())

    # Record results for this year
    for cat in labels:
        mass_val = float(grouped.get(cat, 0.0))
        pct = (mass_val / total_mass * 100.0) if total_mass > 0 else 0.0
        data.append({
            'Year': year,
            'Category': cat,
            'Mass (Gt)': mass_val,
            'Proportion (%)': round(pct, 2)
        })

# Create a DataFrame and sort
df_summary = pd.DataFrame(data)
df_summary = df_summary.sort_values(['Year', 'Category']).reset_index(drop=True)

# Display the summary table
print(df_summary)

# Save to CSV
df_summary.to_csv('iceberg_mass_by_category_per_year.csv', index=False)


    Year      Category    Mass (Gt)  Proportion (%)
0   2018       0-1 km² 1,577.548729       19.980000
1   2018      1-10 km² 1,729.553059       21.910000
2   2018    10-100 km²   922.938900       11.690000
3   2018  100-1000 km²   960.851808       12.170000
4   2018     >1000 km² 2,704.223445       34.250000
5   2019       0-1 km² 1,713.269726       20.010000
6   2019      1-10 km² 1,905.426939       22.250000
7   2019    10-100 km²   973.356776       11.370000
8   2019  100-1000 km²   761.788737        8.900000
9   2019     >1000 km² 3,209.540722       37.480000
10  2020       0-1 km² 1,683.796493       17.870000
11  2020      1-10 km² 2,188.418534       23.230000
12  2020    10-100 km² 1,337.990042       14.200000
13  2020  100-1000 km² 1,404.455599       14.910000
14  2020     >1000 km² 2,805.600163       29.780000
15  2021       0-1 km² 2,040.964467       19.150000
16  2021      1-10 km² 1,838.375736       17.250000
17  2021    10-100 km² 1,168.259949       10.960000
18  2021  10

In [9]:
import os
from glob import glob
import warnings
import pandas as pd
import geopandas as gpd

# Suppress centroid-in-geographic-CRS warnings
warnings.filterwarnings("ignore", "Geometry is in a geographic CRS.*")

# Folder containing the GeoPackage files
gpkg_folder = "Iceberg vector outline"

# Define size bins (km²) and labels
bins = [0, 1, 10, 100, 1000, float('inf')]
size_labels = ['0-1 km²', '1-10 km²', '10-100 km²', '100-1000 km²', '>1000 km²']

# Function to assign sector based on longitude
def assign_sector(lon):
    if lon >= 20 and lon < 90:
        return 'Indian Ocean Sector'
    elif lon >= 90 and lon < 160:
        return 'Western Pacific Ocean Sector'
    elif (lon >= 160 and lon <= 180) or (lon >= -180 and lon < -130):
        return 'Ross Sea Sector'
    elif lon >= -130 and lon < -60:
        return 'Amundsen and Bellingshausen Seas Sector'
    elif lon >= -60 and lon < 20:
        return 'Weddell Sea Sector'
    else:
        return 'Unknown'

# List to collect sector & size summaries
sector_summary = []

# Process each GeoPackage file
gpkg_paths = sorted(glob(os.path.join(gpkg_folder, '*.gpkg')))
for gpkg in gpkg_paths:
    # Extract year from filename (assumes filename starts with YYYY)
    year = int(os.path.basename(gpkg)[:4])
    layer = f"layer_{year}10_distribution"
    gdf = gpd.read_file(gpkg, layer=layer)

    # Assign sector and size categories using new fields
    gdf['Sector'] = gdf['lon'].apply(assign_sector)
    gdf['Size_Category'] = pd.cut(gdf['area_km2'], bins=bins, labels=size_labels, right=False)

    # Summarize by sector & size
    grp = gdf.groupby(['Sector', 'Size_Category'])['mass_gt'].sum().reset_index()
    for _, row in grp.iterrows():
        sector_summary.append({
            'Year': year,
            'Sector': row['Sector'],
            'Size_Category': row['Size_Category'],
            'Mass (Gt)': row['mass_gt']
        })

# Create summary DataFrame and sort
df_sector = pd.DataFrame(sector_summary)
df_sector = df_sector.sort_values(['Year', 'Sector', 'Size_Category']).reset_index(drop=True)

# Display only Sector & Size summary
print("=== Sector & Size Iceberg Mass Summary ===")
print(df_sector)

# Optionally save to Excel or CSV
df_sector.to_csv('iceberg_mass_by_sector_size_summary.csv', index=False)


=== Sector & Size Iceberg Mass Summary ===
     Year                                   Sector Size_Category  Mass (Gt)
0    2018  Amundsen and Bellingshausen Seas Sector       0-1 km² 311.267683
1    2018  Amundsen and Bellingshausen Seas Sector      1-10 km² 405.072775
2    2018  Amundsen and Bellingshausen Seas Sector    10-100 km² 182.594871
3    2018  Amundsen and Bellingshausen Seas Sector  100-1000 km² 107.755750
4    2018  Amundsen and Bellingshausen Seas Sector     >1000 km² 669.949457
..    ...                                      ...           ...        ...
145  2023             Western Pacific Ocean Sector       0-1 km² 656.970953
146  2023             Western Pacific Ocean Sector      1-10 km² 602.504107
147  2023             Western Pacific Ocean Sector    10-100 km² 291.439340
148  2023             Western Pacific Ocean Sector  100-1000 km² 282.185667
149  2023             Western Pacific Ocean Sector     >1000 km²   0.000000

[150 rows x 4 columns]
